#### Error Mitigated QSVM (ZNE + REM Combined) - Spambase

This notebook implements **combined error mitigation** using:
- **Zero-Noise Extrapolation (ZNE)**: Extrapolates results from multiple noise scales
- **Readout Error Mitigation (REM)**: Corrects measurement errors using calibration matrix

Focus: Comparing generalization capability of Error-Mitigated QSVM vs Classical SVM

In [7]:
%pip install qiskit qiskit-machine-learning qiskit-aer

Note: you may need to restart the kernel to use updated packages.


In [8]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 2.2.3
Aer: 0.17.2
QML: 0.9.0


In [9]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [10]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [11]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [12]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our",
    "word_freq_over", "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail",
    "word_freq_receive", "word_freq_will", "word_freq_people", "word_freq_report", "word_freq_addresses",
    "word_freq_free", "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", "word_freq_hp",
    "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85",
    "word_freq_technology", "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", "word_freq_re",
    "word_freq_edu", "word_freq_table", "word_freq_conference", "char_freq_;", "char_freq_(",
    "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
file_path = "/home/azureuser/cloudfiles/code/data/spambase/spambase.data"
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

Dataset loaded: 4210 samples, 58 features


##### Noise Model and Error Mitigation Functions

In [13]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard'):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")

Noise model factory function ready!


In [14]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTION
# ==========================================

def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    
    The correction formula for fidelity with symmetric readout error:
    K_corrected = (K_noisy - bias) / correction_factor
    """
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    # Clip to valid kernel range [0, 1]
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    # Ensure diagonal is exactly 1 (self-similarity)
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel

print("REM function ready!")

REM function ready!


##### Experiment Configurations (ZNE+REM Combined Only)

In [15]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (ZNE+REM COMBINED)
# ==========================================
#
# All experiments use combined ZNE + REM error mitigation
# for best-case error-mitigated QSVM performance.
#
# ==========================================

experiments = [
    # --- EXP 1: Sample Size Effect ---
    {'id': 'Exp1_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_400samp',  'samples': 400, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 2: Dimensionality Effect ---
    {'id': 'Exp2_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_6feat',   'samples': 300, 'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_8feat',   'samples': 300, 'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    # {'id': 'Exp2_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 3: Shot Noise Effect ---
    {'id': 'Exp3_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 4: Reps Effect ---
    {'id': 'Exp4_Reps1', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps2', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps3', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 5: Entanglement Effect ---
    {'id': 'Exp5_Linear',   'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear',   'noise_level': 'standard'},
    {'id': 'Exp5_Circular', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp5_Full',     'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard'},

    # --- EXP 6: Noise Level Effect ---
    {'id': 'Exp6_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    {'id': 'Exp6_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp6_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments configured: {len(experiments)}")
print("All experiments use combined ZNE+REM error mitigation")

Total experiments configured: 22
All experiments use combined ZNE+REM error mitigation


##### Main Experiment Loop (ZNE+REM Combined)

In [ ]:
# ===================================================================
# MAIN EXPERIMENT LOOP - ERROR MITIGATED QSVM (ZNE + REM COMBINED)
# ===================================================================
import os
import time
from sklearn.feature_selection import VarianceThreshold

# Setup kernel directory
kernel_dir = 'kernels_em_znerem_spambase'
os.makedirs(kernel_dir, exist_ok=True)

# ZNE scales for Richardson extrapolation
ZNE_SCALES = [1.0, 3.0]

all_results = []

for exp_num, config in enumerate(experiments, 1):
    print("\n" + "="*80)
    print(f"EXPERIMENT {exp_num}/{len(experiments)}: {config['id']}")
    print("="*80)
    
    # Check if files exist (but DON'T skip the loop!)
    train_file_scale1 = f'{kernel_dir}/kernel_train_scale1_{config["id"]}.npy'
    test_file_scale1 = f'{kernel_dir}/kernel_test_scale1_{config["id"]}.npy'
    train_file_scale3 = f'{kernel_dir}/kernel_train_scale3_{config["id"]}.npy'
    test_file_scale3 = f'{kernel_dir}/kernel_test_scale3_{config["id"]}.npy'
    
    skip_quantum = (os.path.exists(train_file_scale1) and 
                    os.path.exists(test_file_scale1) and
                    os.path.exists(train_file_scale3) and 
                    os.path.exists(test_file_scale3))
    
    # -------------------------------------------------------------------
    # DATA PREPARATION
    # -------------------------------------------------------------------
    X = df.drop('label', axis=1)
    y = df['label']
    
    subset_size = int(round(config['samples'] / 0.7))
    
    # First sample LARGER subset from full dataset
    X_subset, _, y_subset, _ = train_test_split(
        X, y, train_size=subset_size, stratify=y, random_state=42
    )
    
    # Then do 70/30 split to get exactly config['samples'] training samples
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, test_size=0.30, random_state=42, stratify=y_subset
    )
    
    # Scaling
    selector_variance = VarianceThreshold(threshold=0)
    X_train_filtered = selector_variance.fit_transform(X_train)
    X_test_filtered = selector_variance.transform(X_test)
    remaining_cols = X_train.columns[selector_variance.get_support()]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_filtered)
    X_test_scaled = scaler.transform(X_test_filtered)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=remaining_cols)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=remaining_cols)
    
    # Correlation dropping
    THRESH = 0.9
    corr_matrix_train = X_train_scaled_df.corr().abs()
    upper_triangle = corr_matrix_train.where(
        np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool)
    )
    columns_to_drop = set()
    for column in upper_triangle.columns:
        high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
        if high_corr_partners:
            for partner in high_corr_partners:
                corr_main = y_train.corr(X_train_scaled_df[column])
                corr_partner = y_train.corr(X_train_scaled_df[partner])
                if abs(corr_main) < abs(corr_partner):
                    columns_to_drop.add(column)
                else:
                    columns_to_drop.add(partner)
    
    to_drop_final = sorted(list(columns_to_drop))
    X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
    X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)
    
    # SelectKBest
    k_features = config['k_features']
    selector = SelectKBest(score_func=f_classif, k=k_features)
    X_train_kbest = selector.fit_transform(X_train_selected, y_train)
    X_test_kbest = selector.transform(X_test_selected)
    selected_features = X_train_selected.columns[selector.get_support()].tolist()
    
    # -------------------------------------------------------------------
    # KERNEL (Load OR Compute)
    # -------------------------------------------------------------------
    kernel_time = 0
    
    if skip_quantum:
        print(f"Loading existing kernels...")
        kernels_train = {
            1.0: np.load(train_file_scale1),
            3.0: np.load(train_file_scale3)
        }
        kernels_test = {
            1.0: np.load(test_file_scale1),
            3.0: np.load(test_file_scale3)
        }
        noise_level = config.get('noise_level', 'standard')
    else:
        print(f"Computing quantum kernels...")
        noise_level = config.get('noise_level', 'standard')
        
        kernels_train = {}
        kernels_test = {}
        
        feature_map = ZZFeatureMap(
            feature_dimension=k_features, 
            reps=config['reps'], 
            entanglement=config['entanglement']
        )
        
        start_kernel = time.time()
        for scale in ZNE_SCALES:
            _, backend, pm, _ = get_scaled_noise_model(scale_factor=scale, level=noise_level)
            sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
            fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
            qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
            
            kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
            kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
        
        kernel_time = time.time() - start_kernel
        
        np.save(train_file_scale1, kernels_train[1.0])
        np.save(test_file_scale1, kernels_test[1.0])
        np.save(train_file_scale3, kernels_train[3.0])
        np.save(test_file_scale3, kernels_test[3.0])
    
    # -------------------------------------------------------------------
    # ERROR MITIGATION (ALWAYS RUN)
    # -------------------------------------------------------------------
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(noise_level, NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k_features
    
    # Step 1: ZNE (Linear Richardson extrapolation)
    kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
    kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
    
    # Step 2: REM (Readout error mitigation)
    matrix_train = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
    matrix_test = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
    
    # Ensure validity
    matrix_train = np.clip(matrix_train, 0, 1)
    matrix_test = np.clip(matrix_test, 0, 1)
    
    # -------------------------------------------------------------------
    # GRID SEARCH & EVALUATION (ALWAYS RUN)
    # -------------------------------------------------------------------
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        SVC(kernel='precomputed', class_weight='balanced'),
        param_grid, cv=cv, scoring='accuracy', n_jobs=-1
    )
    
    start_train = time.time()
    grid_search.fit(matrix_train, y_train)
    train_time = time.time() - start_train
    
    best_model = grid_search.best_estimator_
    best_c = grid_search.best_params_['C']
    cv_score = grid_search.best_score_
    
    y_train_pred = best_model.predict(matrix_train)
    y_test_pred = best_model.predict(matrix_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
    spam_recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    print(f"Test Acc: {test_acc:.4f} | Recall: {spam_recall:.4f}")
    
    # -------------------------------------------------------------------
    # STORE (ALWAYS)
    # -------------------------------------------------------------------
    all_results.append({
        'experiment_id': config['id'],
        'exp_number': exp_num,
        'samples': config['samples'],
        'k_features': k_features,
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': noise_level,
        'selected_features': selected_features,
        'best_c': best_c,
        'cv_score': cv_score,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_balanced_acc': test_balanced_acc,
        'spam_recall': spam_recall,
        'gen_gap': gen_gap,
        'kernel_time': kernel_time,
        'train_time': train_time
    })

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)

results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_spambase_znerem_results.csv', index=False)
print(f"Saved {len(results_df)} experiments to kernels_em_znerem_spambase/")



EXPERIMENT 1/22: Exp1_100samp (ZNE+REM Mitigated)
Samples: 100 | K Features: 8 | Shots: 1024
Noise Level: standard | Reps: 1 | Entanglement: linear

Data prepared: Train=(100, 8), Test=(43, 8)

Computing quantum kernels...
  → Scale factor 1.0... 

/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_15707/2641917031.py:92: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Done
  → Scale factor 3.0... Done
  → Total kernel time: 527.14s

Applying error mitigation...
  → ZNE: Linear extrapolation [1.0, 3.0]
  → REM: Readout correction (p_ro=0.02, n_qubits=8)

Grid searching for optimal C...
  → Best C: 100
  → CV Score: 0.7493
  → Training time: 2.34s
  → Train Accuracy: 1.0000
  → Test Accuracy: 0.7907
  → Test Balanced Accuracy: 0.8269
  → Spam Recall: 1.0000
  → Generalization Gap: 0.2093

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       1.00      0.65      0.79        26
    Spam (1)       0.65      1.00      0.79        17

    accuracy                           0.79        43
   macro avg       0.83      0.83      0.79        43
weighted avg       0.86      0.79      0.79        43


Experiment 1/22 complete! Total time: 529.53s

EXPERIMENT 2/22: Exp1_200samp (ZNE+REM Mitigated)
Samples: 200 | K Features: 8 | Shots: 1024
Noise Level: standard | Reps: 1 | Entanglement: linear

Data prepared: Train=(200,

/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_15707/2641917031.py:92: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Done
  → Scale factor 3.0... Done
  → Total kernel time: 6103.26s

Applying error mitigation...
  → ZNE: Linear extrapolation [1.0, 3.0]
  → REM: Readout correction (p_ro=0.02, n_qubits=8)

Grid searching for optimal C...
  → Best C: 1
  → CV Score: 0.7604
  → Training time: 2.28s
  → Train Accuracy: 0.9650
  → Test Accuracy: 0.8023
  → Test Balanced Accuracy: 0.8213
  → Spam Recall: 0.9118
  → Generalization Gap: 0.1627

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       0.93      0.73      0.82        52
    Spam (1)       0.69      0.91      0.78        34

    accuracy                           0.80        86
   macro avg       0.81      0.82      0.80        86
weighted avg       0.83      0.80      0.80        86


Experiment 2/22 complete! Total time: 6105.58s

EXPERIMENT 3/22: Exp1_300samp (ZNE+REM Mitigated)
Samples: 300 | K Features: 8 | Shots: 1024
Noise Level: standard | Reps: 1 | Entanglement: linear

Data prepared: Train=(300,

/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/anaconda/envs/jupyter_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_15707/2641917031.py:92: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


##### Results Summary

In [ ]:
# Display summary table
print("\nResults Summary (ZNE+REM Error Mitigated QSVM):")
print(results_df[['experiment_id', 'samples', 'k_features', 'noise_level', 'test_acc', 'spam_recall', 'gen_gap']].to_string(index=False))

In [ ]:
# ==========================================
# FIND BEST CONFIGURATIONS
# ==========================================

print("\n" + "=" * 80)
print("BEST CONFIGURATIONS")
print("=" * 80)

# Best overall test accuracy
best_acc_idx = results_df['test_acc'].idxmax()
best_acc_config = results_df.iloc[best_acc_idx]

print("\n BEST TEST ACCURACY:")
print(f"  Experiment: {best_acc_config['experiment_id']}")
print(f"  Test Accuracy: {best_acc_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_acc_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_acc_config['gen_gap']:.4f}")

# Best spam recall
best_recall_idx = results_df['spam_recall'].idxmax()
best_recall_config = results_df.iloc[best_recall_idx]

print("\n BEST SPAM RECALL:")
print(f"  Experiment: {best_recall_config['experiment_id']}")
print(f"  Test Accuracy: {best_recall_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_recall_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_recall_config['gen_gap']:.4f}")

# Best generalization (lowest gap)
best_gen_idx = results_df['gen_gap'].idxmin()
best_gen_config = results_df.iloc[best_gen_idx]

print("\n BEST GENERALIZATION (Lowest Gap):")
print(f"  Experiment: {best_gen_config['experiment_id']}")
print(f"  Test Accuracy: {best_gen_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_gen_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_gen_config['gen_gap']:.4f}")

##### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (18, 5)

def plot_experiment_group(df, group_prefix, param_col, xlabel, log_x=False):
    """Plot results for a specific experiment group."""
    subset = df[df['experiment_id'].str.contains(group_prefix)].copy()
    if subset.empty:
        print(f"No data for {group_prefix}")
        return
    
    subset = subset.sort_values(param_col)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Performance
    axes[0].plot(subset[param_col], subset['test_acc'], 'o-', label='Test Accuracy', color='#1f77b4')
    axes[0].plot(subset[param_col], subset['spam_recall'], 's--', label='Spam Recall', color='#ff7f0e')
    if log_x: axes[0].set_xscale('log', base=2)
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Score')
    axes[0].set_title(f'Performance vs {xlabel}')
    axes[0].set_ylim(0, 1.05)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Generalization Gap
    axes[1].plot(subset[param_col], subset['gen_gap'], 'D-', color='#d62728')
    if log_x: axes[1].set_xscale('log', base=2)
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel('Gap')
    axes[1].set_title('Generalization Gap (Lower is Better)')
    axes[1].grid(True, alpha=0.3)
    
    # Time
    axes[2].plot(subset[param_col], subset['time_seconds'], '^-', color='#2ca02c')
    if log_x: axes[2].set_xscale('log', base=2)
    axes[2].set_xlabel(xlabel)
    axes[2].set_ylabel('Time (s)')
    axes[2].set_title('Computational Cost')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'{group_prefix} Experiments (ZNE+REM)', fontsize=14)
    plt.tight_layout()
    plt.show()

# Plot each experiment group
plot_experiment_group(results_df, 'Exp1', 'samples', 'Training Samples')
plot_experiment_group(results_df, 'Exp2', 'k_features', 'Feature Dimension (Qubits)')
plot_experiment_group(results_df, 'Exp3', 'shots', 'Shots', log_x=True)

In [ ]:
# ==========================================
# HEATMAP: All Experiments Overview
# ==========================================

plt.figure(figsize=(12, 8))

heatmap_data = results_df.set_index('experiment_id')[['test_acc', 'spam_recall', 'gen_gap', 'train_acc', 'cv_score']]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=0.75, linewidths=.5, cbar_kws={'label': 'Score'})

plt.title('ZNE+REM Error Mitigated QSVM - Spambase', fontsize=14, pad=20)
plt.ylabel('Experiment ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('em_qsvm_spambase_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()